In [14]:
import sys
import time
import config
import re 
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from sklearn.preprocessing import MultiLabelBinarizer


In [15]:
url = config.urlbasecsv
driver = webdriver.Chrome()
driver.get(url)


In [16]:

tabela_remedios = WebDriverWait(driver, 20).until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, '#conteudo > div.ato > table:nth-child(21)'))
)

time.sleep(2)



In [17]:
linhas = driver.find_elements(By.CSS_SELECTOR, "table tr")

dados = []
for linha in linhas:
    colunas = [td.text.strip() for td in linha.find_elements(By.TAG_NAME, "td")]
    if colunas:  
        dados.append(colunas)

df = pd.DataFrame(dados)

df.columns = df.iloc[0]
df = df.drop(df.index[0]).reset_index(drop=True)
df.head(30)



,Fármaco,Subgrupo terapêutico ou farmacológico,Forma farmacêutica,Concentração máxima,Indicação terapêutica simplificada
0,Aceclofenaco,M02A – Produtos para dor articular e muscular ...,Creme dermatológico,15 mg/g,Dor e inflamação do sistema musculoesquelético.
1,Acetato de hidrocortisona,D07A – Corticosteroides (isolados) de uso local,"Creme dermatológico, pomada dermatológica",10 mg/g,"Dermatites, eczemas, eritema solar, queimadura..."
2,Acetilcisteína,"R05C – Expectorantes, excluindo associações co...","Comprimido efervescente, granulado",600 mg,Secreções mucosas densas e viscosas nas vias r...
3,Acetilcisteína,R01A – Descongestionantes e outras preparações...,Solução nasal,"11,5 mg/mL",Congestão nasal associada a rinites ou após pr...
4,Acetilcisteína,"R05C – Expectorantes, excluindo associações co...",Solução oral,40 mg/mL,Secreções mucosas densas e viscosas nas vias r...
5,Acetilracemetionina + citrato de colina + betaína,"A05B – Terapia biliar, lipotrópicos",Solução oral,40 + 53 + 50 mg/mL,Auxiliar no tratamento dos distúrbios do fígado
6,Ácido acetilsalicílico,N02B – Analgésicos e antipiréticos,Comprimido,500 mg,"Febre. Dores leves a moderadas, incluindo as a..."
7,Ácido acetilsalicílico,N02B – Analgésicos e antipiréticos,Comprimido revestido de liberação prolongada,500 mg,"Febre. Dores leves a moderadas, incluindo as a..."
8,Ácido acetilsalicílico + ácido ascórbico (Vit. C),N02B – Analgésicos e antipiréticos,Comprimido,400 + 200 mg,"Sintomas da gripe e resfriado comuns, como feb..."
9,Ácido acetilsalicílico + ácido ascórbico (Vit. C),N02B – Analgésicos e antipiréticos,Comprimido efervescente,400 + 240 mg,"Sintomas da gripe e resfriado comuns, como feb..."


In [18]:
df = df[df["Indicação terapêutica simplificada"].notna()].reset_index(drop=True)


In [19]:
def normalizar_indicacao(texto):
    if not isinstance(texto, str):
        return ""

    # tira quebras de linha
    texto = texto.replace("\n", " ").replace("\r", " ")

    # espaços múltiplos -> 1
    texto = re.sub(r"\s+", " ", texto).strip()

    # TROCA vírgula OU ponto por " | "
    # (onde tiver "," ou ".", entra um pipe)
    texto = re.sub(r"[.,]\s*", " | ", texto)

    # pipes duplicados -> apenas um
    texto = re.sub(r"\|\s*\|+", " | ", texto)

    # tira pipe no começo/fim
    texto = texto.strip(" |")

    return texto


In [20]:

df["indicacao_limpa"] = df["Indicação terapêutica simplificada"].apply(normalizar_indicacao)

In [21]:
import re
df["indicacoes_list"] = df["indicacao_limpa"].apply(
    lambda txt: re.split(r"\s*\|\s*", txt) if isinstance(txt, str) and txt.strip() != "" else []
)

# limpar itens vazios
df["indicacoes_list"] = df["indicacoes_list"].apply(
    lambda lista: [s.strip() for s in lista if s.strip() != ""]
)

In [22]:
df["indicacoes_list"].head()

type(df["indicacoes_list"].iloc[0])


list

In [23]:
#one hot agoraaaaa
mlb = MultiLabelBinarizer()

dummies = pd.DataFrame(
    mlb.fit_transform(df["indicacoes_list"]),
    columns=mlb.classes_,
    index=df.index
    ).astype(int)

df_final = pd.concat([df, dummies], axis=1)

df_final.head(10)


,Fármaco,Subgrupo terapêutico ou farmacológico,Forma farmacêutica,Concentração máxima,Indicação terapêutica simplificada,indicacao_limpa,indicacoes_list,Acne vulgar,Afecções cutâneas,Aftas e outras inflamações da mucosa da boca,...,sem catarro,sem catarro associada a gripes e resfriados ou à inalação de agentes irritantes,sob os seios ou em outras áreas da pele que sofrem atrito em crianças e adultos,teníase,tosse e dor muscular associadas à gripes e resfriados,tricuríase,urticárias,vulvar e peniana,vômito e distensão abdominal,Úlcera cutânea
0,Aceclofenaco,M02A – Produtos para dor articular e muscular ...,Creme dermatológico,15 mg/g,Dor e inflamação do sistema musculoesquelético.,Dor e inflamação do sistema musculoesquelético,[Dor e inflamação do sistema musculoesquelético],0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Acetato de hidrocortisona,D07A – Corticosteroides (isolados) de uso local,"Creme dermatológico, pomada dermatológica",10 mg/g,"Dermatites, eczemas, eritema solar, queimadura...",Dermatites | eczemas | eritema solar | queimad...,"[Dermatites, eczemas, eritema solar, queimadur...",0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Acetilcisteína,"R05C – Expectorantes, excluindo associações co...","Comprimido efervescente, granulado",600 mg,Secreções mucosas densas e viscosas nas vias r...,Secreções mucosas densas e viscosas nas vias r...,[Secreções mucosas densas e viscosas nas vias ...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Acetilcisteína,R01A – Descongestionantes e outras preparações...,Solução nasal,"11,5 mg/mL",Congestão nasal associada a rinites ou após pr...,Congestão nasal associada a rinites ou após pr...,[Congestão nasal associada a rinites ou após p...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Acetilcisteína,"R05C – Expectorantes, excluindo associações co...",Solução oral,40 mg/mL,Secreções mucosas densas e viscosas nas vias r...,Secreções mucosas densas e viscosas nas vias r...,[Secreções mucosas densas e viscosas nas vias ...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,Acetilracemetionina + citrato de colina + betaína,"A05B – Terapia biliar, lipotrópicos",Solução oral,40 + 53 + 50 mg/mL,Auxiliar no tratamento dos distúrbios do fígado,Auxiliar no tratamento dos distúrbios do fígado,[Auxiliar no tratamento dos distúrbios do fígado],0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,Ácido acetilsalicílico,N02B – Analgésicos e antipiréticos,Comprimido,500 mg,"Febre. Dores leves a moderadas, incluindo as a...",Febre | Dores leves a moderadas | incluindo as...,"[Febre, Dores leves a moderadas, incluindo as ...",0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,Ácido acetilsalicílico,N02B – Analgésicos e antipiréticos,Comprimido revestido de liberação prolongada,500 mg,"Febre. Dores leves a moderadas, incluindo as a...",Febre | Dores leves a moderadas | incluindo as...,"[Febre, Dores leves a moderadas, incluindo as ...",0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,Ácido acetilsalicílico + ácido ascórbico (Vit. C),N02B – Analgésicos e antipiréticos,Comprimido,400 + 200 mg,"Sintomas da gripe e resfriado comuns, como feb...",Sintomas da gripe e resfriado comuns | como fe...,"[Sintomas da gripe e resfriado comuns, como fe...",0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,Ácido acetilsalicílico + ácido ascórbico (Vit. C),N02B – Analgésicos e antipiréticos,Comprimido efervescente,400 + 240 mg,"Sintomas da gripe e resfriado comuns, como feb...",Sintomas da gripe e resfriado comuns | como fe...,"[Sintomas da gripe e resfriado comuns, como fe...",0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [24]:
colunas = list(dummies.columns)
for c in colunas:
    print(c)

Acne vulgar
Afecções cutâneas
Aftas e outras inflamações da mucosa da boca
Aftas e outras inflamações na mucosa da boca
Alívio da azia
Alívio da azia devido à má digestão
Alívio da azia e dor de estômago devido à má digestão
Alívio da dor de dente
Alívio da regurgitação ácida
Alívio de cólicas gastrintestinais
Alívio de sintomas frequentes (que ocorrem por dois ou mais dias na semana) de azia e queimação
Alívio dos deconfortos bucais da primeira dentição
Ascaridíase
Auxiliar no processo de mineralização óssea
Auxiliar no tratamento dos distúrbios do fígado
Brotoeja
Brotoejas
Candidíase vaginal
Candidíase vaginal e perianal
Candisíase vaginal e peniana
Caspa
Cefaleia
Cefaleia e alergias
Cefaleia e enxaqueca
Coadjuvante no tratamento da diarreia; coadjuvante em enterocolites crônicas e agudas
Congestão nasal
Congestão nasal associada a rinites ou após procedimentos cirúrgicos no nariz
Congestão nasal e tosse associadas a gripes e resfriados
Conjutivite alérgica
Constipação intestinal ou 

In [25]:
mapeamento_colunas = {col: col for col in dummies.columns}
print(mapeamento_colunas)

{'Acne vulgar': 'Acne vulgar', 'Afecções cutâneas': 'Afecções cutâneas', 'Aftas e outras inflamações da mucosa da boca': 'Aftas e outras inflamações da mucosa da boca', 'Aftas e outras inflamações na mucosa da boca': 'Aftas e outras inflamações na mucosa da boca', 'Alívio da azia': 'Alívio da azia', 'Alívio da azia devido à má digestão': 'Alívio da azia devido à má digestão', 'Alívio da azia e dor de estômago devido à má digestão': 'Alívio da azia e dor de estômago devido à má digestão', 'Alívio da dor de dente': 'Alívio da dor de dente', 'Alívio da regurgitação ácida': 'Alívio da regurgitação ácida', 'Alívio de cólicas gastrintestinais': 'Alívio de cólicas gastrintestinais', 'Alívio de sintomas frequentes (que ocorrem por dois ou mais dias na semana) de azia e queimação': 'Alívio de sintomas frequentes (que ocorrem por dois ou mais dias na semana) de azia e queimação', 'Alívio dos deconfortos bucais da primeira dentição': 'Alívio dos deconfortos bucais da primeira dentição', 'Ascaridí

In [26]:
import unidecode

In [27]:
import re

def encode_nome(s: str) -> str:
    s = s.lower().strip()
    s = unidecode.unidecode(s)            # tira acentos
    s = re.sub(r'[^a-z0-9]+', '_', s)     # tudo que não é letra/numero vira _
    s = re.sub(r'_+', '_', s)             # colapsa vários ___ em _
    s = s.strip('_')                      # tira _ do começo/fim
    return s 

# resumindo  snake_case


In [28]:
edit_colunas = {orig: encode_nome(orig) for orig in mapeamento_colunas.keys()}


In [29]:
dummies = dummies.rename(columns=edit_colunas)

In [30]:
print(dummies)

     acne_vulgar  afeccoes_cutaneas  \
0              0                  0   
1              0                  0   
2              0                  0   
3              0                  0   
4              0                  0   
..           ...                ...   
259            0                  0   
260            0                  0   
261            0                  0   
262            0                  0   
263            0                  0   

     aftas_e_outras_inflamacoes_da_mucosa_da_boca  \
0                                               0   
1                                               0   
2                                               0   
3                                               0   
4                                               0   
..                                            ...   
259                                             0   
260                                             0   
261                                             0   
2

In [31]:
df_final = pd.concat([df, dummies], axis=1)

In [32]:
df_final.head(30)

,Fármaco,Subgrupo terapêutico ou farmacológico,Forma farmacêutica,Concentração máxima,Indicação terapêutica simplificada,indicacao_limpa,indicacoes_list,acne_vulgar,afeccoes_cutaneas,aftas_e_outras_inflamacoes_da_mucosa_da_boca,...,sem_catarro,sem_catarro_associada_a_gripes_e_resfriados_ou_a_inalacao_de_agentes_irritantes,sob_os_seios_ou_em_outras_areas_da_pele_que_sofrem_atrito_em_criancas_e_adultos,teniase,tosse_e_dor_muscular_associadas_a_gripes_e_resfriados,tricuriase,urticarias,vulvar_e_peniana,vomito_e_distensao_abdominal,ulcera_cutanea
0,Aceclofenaco,M02A – Produtos para dor articular e muscular ...,Creme dermatológico,15 mg/g,Dor e inflamação do sistema musculoesquelético.,Dor e inflamação do sistema musculoesquelético,[Dor e inflamação do sistema musculoesquelético],0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Acetato de hidrocortisona,D07A – Corticosteroides (isolados) de uso local,"Creme dermatológico, pomada dermatológica",10 mg/g,"Dermatites, eczemas, eritema solar, queimadura...",Dermatites | eczemas | eritema solar | queimad...,"[Dermatites, eczemas, eritema solar, queimadur...",0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Acetilcisteína,"R05C – Expectorantes, excluindo associações co...","Comprimido efervescente, granulado",600 mg,Secreções mucosas densas e viscosas nas vias r...,Secreções mucosas densas e viscosas nas vias r...,[Secreções mucosas densas e viscosas nas vias ...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Acetilcisteína,R01A – Descongestionantes e outras preparações...,Solução nasal,"11,5 mg/mL",Congestão nasal associada a rinites ou após pr...,Congestão nasal associada a rinites ou após pr...,[Congestão nasal associada a rinites ou após p...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Acetilcisteína,"R05C – Expectorantes, excluindo associações co...",Solução oral,40 mg/mL,Secreções mucosas densas e viscosas nas vias r...,Secreções mucosas densas e viscosas nas vias r...,[Secreções mucosas densas e viscosas nas vias ...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,Acetilracemetionina + citrato de colina + betaína,"A05B – Terapia biliar, lipotrópicos",Solução oral,40 + 53 + 50 mg/mL,Auxiliar no tratamento dos distúrbios do fígado,Auxiliar no tratamento dos distúrbios do fígado,[Auxiliar no tratamento dos distúrbios do fígado],0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,Ácido acetilsalicílico,N02B – Analgésicos e antipiréticos,Comprimido,500 mg,"Febre. Dores leves a moderadas, incluindo as a...",Febre | Dores leves a moderadas | incluindo as...,"[Febre, Dores leves a moderadas, incluindo as ...",0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,Ácido acetilsalicílico,N02B – Analgésicos e antipiréticos,Comprimido revestido de liberação prolongada,500 mg,"Febre. Dores leves a moderadas, incluindo as a...",Febre | Dores leves a moderadas | incluindo as...,"[Febre, Dores leves a moderadas, incluindo as ...",0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,Ácido acetilsalicílico + ácido ascórbico (Vit. C),N02B – Analgésicos e antipiréticos,Comprimido,400 + 200 mg,"Sintomas da gripe e resfriado comuns, como feb...",Sintomas da gripe e resfriado comuns | como fe...,"[Sintomas da gripe e resfriado comuns, como fe...",0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,Ácido acetilsalicílico + ácido ascórbico (Vit. C),N02B – Analgésicos e antipiréticos,Comprimido efervescente,400 + 240 mg,"Sintomas da gripe e resfriado comuns, como feb...",Sintomas da gripe e resfriado comuns | como fe...,"[Sintomas da gripe e resfriado comuns, como fe...",0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [33]:
df_final.to_csv("tabela_anvisa_limpa.csv", index=False, sep=";", encoding="utf-8-sig")


## Inserindo os dados no banco

In [38]:
import pandas as pd
import unicodedata
from sqlalchemy import create_engine, text
from sqlalchemy.orm import Session
import config

engine = create_engine(config.conexao_banco)

# ---------------------------
# NORMALIZAÇÃO
# ---------------------------

def normalizar(txt):
    if pd.isna(txt):
        return None
    txt = unicodedata.normalize("NFKD", str(txt))
    txt = txt.encode("ASCII", "ignore").decode("ASCII")
    return txt.strip().title()

# ---------------------------
# GET OR CREATE
# ---------------------------

def get_or_create_sintoma(sessao, nome):
    nome = normalizar(nome)

    row = sessao.execute(
        text("SELECT idsintoma FROM sintoma WHERE nomeSintoma = :n"),
        {"n": nome}
    ).fetchone()

    if row:
        return row[0]

    res = sessao.execute(
        text("INSERT INTO sintoma (nomeSintoma) VALUES (:n)"),
        {"n": nome}
    )
    return res.lastrowid


def get_or_create_substancia(sessao, nome, subgrupo):
    nome = normalizar(nome)
    subgrupo = normalizar(subgrupo)

    row = sessao.execute(
        text("""
            SELECT idsubstancia, sugrupo
            FROM substancia
            WHERE nomeSubstancia = :n
        """),
        {"n": nome}
    ).fetchone()

    # já existe
    if row:
        id_sub, sugrupo_atual = row

        # se ainda não tem subgrupo salvo, atualiza
        if not sugrupo_atual and subgrupo:
            sessao.execute(
                text("""
                    UPDATE substancia
                    SET sugrupo = :s
                    WHERE idsubstancia = :id
                """),
                {"s": subgrupo, "id": id_sub}
            )

        return id_sub

    # não existe → cria já com subgrupo
    res = sessao.execute(
        text("""
            INSERT INTO substancia (nomeSubstancia, sugrupo)
            VALUES (:n, :s)
        """),
        {"n": nome, "s": subgrupo}
    )
    return res.lastrowid



def relacionar(sessao, idsintoma, idsubstancia):
    existe = sessao.execute(
        text("""
            SELECT 1
            FROM sintoma_has_substancia
            WHERE idsintoma = :s AND idsubstancia = :sub
        """),
        {"s": idsintoma, "sub": idsubstancia}
    ).fetchone()

    if not existe:
        sessao.execute(
            text("""
                INSERT INTO sintoma_has_substancia (idsintoma, idsubstancia)
                VALUES (:s, :sub)
            """),
            {"s": idsintoma, "sub": idsubstancia}
        )

# ---------------------------
# IMPORTAÇÃO CSV
# ---------------------------

df = pd.read_csv(
    "/home/mgabriel4/Documentos/GitHub/inter-4sem-2025-medicamentos/tabela_anvisa_limpa.csv",
    sep=";"
)

# coluna que representa a substância
coluna_substancia = "Fármaco"

# colunas que NÃO são sintomas
colunas_descartar = {
    "Fármaco",
    "Subgrupo terapêutico ou farmacológico",
    "Forma farmacêutica",
    "Concentração máxima",
    "Indicação terapêutica simplificada",
    "indicacao_limpa",
    "indicacoes_list"
}

# sintomas = todas as colunas que sobraram
sintomas = [c for c in df.columns if c not in colunas_descartar]

print(f"🧪 Total de sintomas detectados: {len(sintomas)}")

with Session(engine) as sessao, sessao.begin():
    for _, linha in df.iterrows():
        nome_sub = linha[coluna_substancia]
        subgrupo = linha["Subgrupo terapêutico ou farmacológico"]

        id_sub = get_or_create_substancia(sessao, nome_sub, subgrupo)

        for sintoma in sintomas:
            valor = linha[sintoma]

            if valor == 1:
                id_s = get_or_create_sintoma(sessao, sintoma)
                relacionar(sessao, id_s, id_sub)

print("✅ Importação concluída com sucesso!")


🧪 Total de sintomas detectados: 210
✅ Importação concluída com sucesso!
